## 1. KIPRIS에서 NFVQ 200건씩 가져오기

In [6]:
import os
import requests
import xml.etree.ElementTree as ET
import time
import gc # 가비지 컬렉션 (파일 핸들 해제용)
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

# ==========================================
# 1. 설정 변수
# ==========================================
KIPRIS_API_KEY = '키'
ROOT_DRIVE_FOLDER_ID = '파이널'

QUERIES = ['G06N', 'G06F', 'G06V', 'G06Q']
TARGET_COUNT = 200

SCOPES = ['https://www.googleapis.com/auth/drive.file']
TEMP_DIR = './temp_pdfs'
if not os.path.exists(TEMP_DIR): os.makedirs(TEMP_DIR)

folder_id_cache = {}

# ==========================================
# 2. 유틸리티 함수
# ==========================================
def get_drive_service():
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return build('drive', 'v3', credentials=creds)

def get_subfolder_id(service, folder_name):
    if folder_name in folder_id_cache: return folder_id_cache[folder_name]
    query = f"name = '{folder_name}' and '{ROOT_DRIVE_FOLDER_ID}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    results = service.files().list(q=query).execute()
    items = results.get('files', [])
    if items:
        folder_id = items[0]['id']
    else:
        file_metadata = {'name': folder_name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [ROOT_DRIVE_FOLDER_ID]}
        folder = service.files().create(body=file_metadata, fields='id').execute()
        folder_id = folder.get('id')
    folder_id_cache[folder_name] = folder_id
    return folder_id

# ==========================================
# 3. 메인 실행 로직
# ==========================================
def fetch_and_upload_patents():
    drive_service = get_drive_service()
    
    SEARCH_URL = 'http://plus.kipris.or.kr/openapi/rest/patUtiModInfoSearchSevice/ipcSearchInfo'
    DETAIL_URL = 'http://plus.kipris.or.kr/kipo-api/kipi/patUtiModInfoSearchSevice/getAnnFullTextInfoSearch'
    
    for query in QUERIES:
        print(f"\n>>> [{query}] 작업 시작")
        target_folder_id = get_subfolder_id(drive_service, query)
        fetched_count = 0
        docs_start = 1
        
        while fetched_count < TARGET_COUNT:
            params = {'ipcNumber': query, 'docsStart': docs_start, 'accessKey': KIPRIS_API_KEY}
            
            try:
                res = requests.get(SEARCH_URL, params=params)
                root = ET.fromstring(res.content)
                items = root.findall('.//PatentUtilityInfo')
                
                if not items:
                    print(f"    [정보] 더 이상 검색 결과가 없습니다.")
                    break

                for item in items:
                    if fetched_count >= TARGET_COUNT: break
                    
                    app_num = item.findtext('ApplicationNumber')
                    if not app_num: continue
                    
                    # 2단계: PDF 경로 가져오기
                    detail_res = requests.get(DETAIL_URL, params={'applicationNumber': app_num, 'ServiceKey': KIPRIS_API_KEY})
                    detail_root = ET.fromstring(detail_res.content)
                    pdf_path = detail_root.findtext('.//path')
                    
                    if not pdf_path:
                        # 경로가 없는 경우 다음 특허로 바로 넘어감
                        continue 
                    
                    file_name = f"{app_num}.pdf"
                    local_path = os.path.abspath(os.path.join(TEMP_DIR, file_name))
                    
                    try:
                        # PDF 다운로드
                        pdf_file_res = requests.get(pdf_path, timeout=30)
                        if pdf_file_res.status_code == 200:
                            with open(local_path, 'wb') as f:
                                f.write(pdf_file_res.content)
                            
                            # 구글 드라이브 업로드
                            media = MediaFileUpload(local_path, mimetype='application/pdf', resumable=True)
                            drive_service.files().create(
                                body={'name': file_name, 'parents': [target_folder_id]},
                                media_body=media
                            ).execute()
                            
                            # 파일 핸들 강제 해제 및 삭제 (WinError 32 방지)
                            del media 
                            gc.collect() 
                            time.sleep(0.5) # 시스템이 파일을 놓아줄 시간을 줌
                            
                            if os.path.exists(local_path):
                                os.remove(local_path)
                            
                            fetched_count += 1
                            print(f"    ({fetched_count}/{TARGET_COUNT}) 업로드 완료: {app_num}")
                    except Exception as upload_err:
                        print(f"    (에러) {app_num} 처리 중 문제 발생: {upload_err}")
                        if os.path.exists(local_path):
                            try: os.remove(local_path)
                            except: pass
                    
                    time.sleep(0.1) # 서버 매너 타임
                
                docs_start += len(items)
                
            except Exception as e:
                print(f"    [중단] {query} 처리 중 에러: {e}")
                break

if __name__ == '__main__':
    fetch_and_upload_patents()


>>> [G06N] 작업 시작
    (1/200) 업로드 완료: 1020210168032
    (2/200) 업로드 완료: 1020230134631
    (3/200) 업로드 완료: 1020200126707
    (4/200) 업로드 완료: 1020170144234
    (5/200) 업로드 완료: 1020250155326
    (6/200) 업로드 완료: 1020240173044
    (7/200) 업로드 완료: 1020210127454
    (8/200) 업로드 완료: 1020240058416
    (9/200) 업로드 완료: 1020200001714
    (10/200) 업로드 완료: 1020217023623
    (11/200) 업로드 완료: 1020207032320
    (12/200) 업로드 완료: 1020190101984
    (13/200) 업로드 완료: 1020177006950
    (14/200) 업로드 완료: 1020197012085
    (에러) 1020230118745 처리 중 문제 발생: [WinError 32] 다른 프로세스가 파일을 사용 중이기 때문에 프로세스가 액세스 할 수 없습니다: 'C:\\Users\\Playdata2\\Untitled Folder\\temp_pdfs\\1020230118745.pdf'
    (15/200) 업로드 완료: 1020230131860
    (16/200) 업로드 완료: 1020230118769
    (17/200) 업로드 완료: 1020210085431
    (18/200) 업로드 완료: 1020230040630
    (19/200) 업로드 완료: 1020220147688
    (20/200) 업로드 완료: 1020220104033
    (21/200) 업로드 완료: 1020230061000
    (22/200) 업로드 완료: 1020210068892
    (23/200) 업로드 완료: 1020170122363
    (24/200) 업로드 완료: 10

In [12]:
import requests

# 가장 단순한 검색 요청을 하나 날려봅니다.
url = 'http://plus.kipris.or.kr/kipo-api/kipi/patUtiModInfoSearchSevice/getWordSearch'
params = {'word': '센서', 'ServiceKey': 'MipgpGJ9pEnSt2ROtI5J0zUkGIFdgIaK02YuQG=TI/Y='}

res = requests.get(url, params=params)

# KIPRIS 서버가 보내는 원본 메시지 전체를 그대로 출력합니다.
print(res.text)

<response>
<header>
<requestMsgID/>
<responseTime>2026-05-08 10:56:28</responseTime>
<responseMsgID/>
<successYN>N</successYN>
<resultCode>22</resultCode>
<resultMsg>LIMITED_NUMBER_OF_SERVICE_REQUESTS_EXCEEDS_ERROR</resultMsg>
</header>
</response>



In [13]:
import os
import requests
import xml.etree.ElementTree as ET
import time
import gc
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

# ==========================================
# 1. 설정 변수
# ==========================================
KIPRIS_API_KEY = 'njb3W6rBi9ZC=TAiEI6/7Ux=JAOUC=68r/KDYNBeU54='
ROOT_DRIVE_FOLDER_ID = '1V-KJTNLjYpxqp_VAgIxKYQO6pm8-zMa2'

# 항목별 검색의 ipcNumber 파라미터에 들어갈 값
QUERIES = ['G06V', 'G06Q']
TARGET_COUNT = 200

SCOPES = ['https://www.googleapis.com/auth/drive.file']
TEMP_DIR = './temp_pdfs'
if not os.path.exists(TEMP_DIR): os.makedirs(TEMP_DIR)

folder_id_cache = {}

# ==========================================
# 2. 구글 인증 및 서비스 생성
# ==========================================
def get_drive_service():
    creds = None
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return build('drive', 'v3', credentials=creds)

def get_subfolder_id(service, folder_name):
    query = f"name = '{folder_name}' and '{ROOT_DRIVE_FOLDER_ID}' in parents and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    results = service.files().list(q=query).execute()
    items = results.get('files', [])
    if items:
        return items[0]['id']
    else:
        file_metadata = {'name': folder_name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [ROOT_DRIVE_FOLDER_ID]}
        folder = service.files().create(body=file_metadata, fields='id').execute()
        return folder.get('id')

# ==========================================
# 3. 메인 실행 로직 (getAdvancedSearch 사용)
# ==========================================
def fetch_and_upload_patents():
    drive_service = get_drive_service()
    
    # 찾아주신 항목별 검색 API 주소 반영
    SEARCH_URL = 'http://plus.kipris.or.kr/kipo-api/kipi/patUtiModInfoSearchSevice/getAdvancedSearch'
    DETAIL_URL = 'http://plus.kipris.or.kr/kipo-api/kipi/patUtiModInfoSearchSevice/getAnnFullTextInfoSearch'
    
    for query in QUERIES:
        print(f"\n==================================================")
        print(f">>> [{query}] 작업 시작 (getAdvancedSearch 방식)")
        print(f"==================================================")
        
        target_folder_id = get_subfolder_id(drive_service, query)
        fetched_count = 0
        page_no = 1
        
        while fetched_count < TARGET_COUNT:
            # 명세서의 요청 파라미터(Request Parameter) 구조 완벽 적용
            params = {
                'ipcNumber': query, # 전용 파라미터 사용!
                'ServiceKey': KIPRIS_API_KEY,
                'numOfRows': '50',
                'pageNo': str(page_no)
            }
            
            try:
                res = requests.get(SEARCH_URL, params=params)
                root = ET.fromstring(res.content)
                items = root.findall('.//item')
                
                if not items:
                    print(f"    [정보] '{query}'에 대한 더 이상 검색 결과가 없습니다.")
                    break

                for item in items:
                    if fetched_count >= TARGET_COUNT: break
                    
                    # 명세서의 출력값(Response Parameter) 구조 적용
                    reg_status = item.findtext('registerStatus')
                    app_num = item.findtext('applicationNumber')
                    
                    # '등록' 상태인 특허만 확실하게 필터링
                    if reg_status != '등록':
                        continue 
                    
                    # 2단계: PDF 경로 가져오기
                    detail_res = requests.get(DETAIL_URL, params={'applicationNumber': app_num, 'ServiceKey': KIPRIS_API_KEY})
                    detail_root = ET.fromstring(detail_res.content)
                    pdf_path = detail_root.findtext('.//path')
                    
                    if not pdf_path: continue 
                    
                    file_name = f"{app_num}.pdf"
                    local_path = os.path.abspath(os.path.join(TEMP_DIR, file_name))
                    
                    try:
                        pdf_file_res = requests.get(pdf_path, timeout=30)
                        if pdf_file_res.status_code == 200:
                            with open(local_path, 'wb') as f:
                                f.write(pdf_file_res.content)
                            
                            media = MediaFileUpload(local_path, mimetype='application/pdf')
                            drive_service.files().create(
                                body={'name': file_name, 'parents': [target_folder_id]},
                                media_body=media
                            ).execute()
                            
                            del media 
                            gc.collect() 
                            time.sleep(0.5)
                            if os.path.exists(local_path): os.remove(local_path)
                            
                            fetched_count += 1
                            print(f"    ({fetched_count}/{TARGET_COUNT}) 업로드 완료: {app_num}")
                    except Exception as e:
                        print(f"    (에러) {app_num} 업로드 실패: {e}")
                
                page_no += 1
                
            except Exception as e:
                print(f"    [중단] 에러 발생: {e}")
                break

if __name__ == '__main__':
    fetch_and_upload_patents()
    print("\n✅ V와 Q 수집 작업이 최종 완료되었습니다!")


>>> [G06V] 작업 시작 (getAdvancedSearch 방식)
    (1/200) 업로드 완료: 1020200004904
    (2/200) 업로드 완료: 1020220189350
    (3/200) 업로드 완료: 1020250096518
    (4/200) 업로드 완료: 1020250065868
    (5/200) 업로드 완료: 1020207029856
    (6/200) 업로드 완료: 1020207034340
    (7/200) 업로드 완료: 1020240094169
    (8/200) 업로드 완료: 1020197006033
    (9/200) 업로드 완료: 1020210060774
    (10/200) 업로드 완료: 1020230048172
    (11/200) 업로드 완료: 1020230067164
    (12/200) 업로드 완료: 1020250172807
    (13/200) 업로드 완료: 1020220143407
    (14/200) 업로드 완료: 1020207027906
    (15/200) 업로드 완료: 1020207013827
    (16/200) 업로드 완료: 1020220181263
    (17/200) 업로드 완료: 1020230099067
    (18/200) 업로드 완료: 1020187009807
    (19/200) 업로드 완료: 1020197021867
    (20/200) 업로드 완료: 1020227020773
    (21/200) 업로드 완료: 1020240150698
    (22/200) 업로드 완료: 1020227021931
    (23/200) 업로드 완료: 1020250162568
    (24/200) 업로드 완료: 1020177021351
    (25/200) 업로드 완료: 1020250000079
    (26/200) 업로드 완료: 1020240062790
    (27/200) 업로드 완료: 1020250060655
    (28/200) 업로드 완료: 10